In [1]:
from google.colab import files
uploaded = files.upload()

Saving employees.csv to employees.csv
Saving logins.txt to logins.txt
Saving orders.json to orders.json
Saving sales.csv to sales.csv


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("OrdersAnalysis").getOrCreate()

Read the text files and print all names

In [3]:
df_txt = spark.read.text("logins.txt")
df_txt.show()

+-----+
|value|
+-----+
|Rahul|
|Sneha|
|Arjun|
|Rahul|
|Priya|
|Sneha|
|Rahul|
|Karan|
|Arjun|
|Sneha|
|Rahul|
| Amit|
|Priya|
|Karan|
|Sneha|
|Rahul|
|Meera|
|Arjun|
|Sneha|
|Rahul|
+-----+
only showing top 20 rows


Count the total login events.

In [4]:
df_txt.count()

26

Unique users

In [5]:
df_txt.select("value").distinct().show()

+-----+
|value|
+-----+
|Meera|
|Sneha|
|Priya|
|Rahul|
|Arjun|
| Amit|
|Karan|
+-----+



Count per user

In [6]:
df_txt.groupBy("value").count().show()

+-----+-----+
|value|count|
+-----+-----+
|Meera|    1|
|Sneha|    6|
|Priya|    3|
|Rahul|    7|
|Arjun|    4|
| Amit|    2|
|Karan|    3|
+-----+-----+



Top 3 users

In [7]:
df_txt.groupBy("value") \
      .count() \
      .orderBy(desc("count")) \
      .show(3)

+-----+-----+
|value|count|
+-----+-----+
|Rahul|    7|
|Sneha|    6|
|Arjun|    4|
+-----+-----+
only showing top 3 rows


Users > 4 logins

In [8]:
df_txt.groupBy("value") \
      .count() \
      .filter(col("count") > 4) \
      .show()

+-----+-----+
|value|count|
+-----+-----+
|Sneha|    6|
|Rahul|    7|
+-----+-----+



Convert to dictionary

In [9]:
dict_counts = {row['value']: row['count'] for row in df_txt.groupBy("value").count().collect()}
print(dict_counts)

{'Meera': 1, 'Sneha': 6, 'Priya': 3, 'Rahul': 7, 'Arjun': 4, 'Amit': 2, 'Karan': 3}


# DATASET 2 — CSV (employees.csv)

Load CSV

In [10]:
emp_df = spark.read.csv("employees.csv", header=True, inferSchema=True)
emp_df.show()

+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|     1| Rahul|        IT| 70000|Hyderabad|
|     2| Sneha|        HR| 60000|Bangalore|
|     3| Arjun|        IT| 75000|  Chennai|
|     4| Priya|   Finance| 80000|Hyderabad|
|     5| Karan|        IT| 50000|   Mumbai|
|     6|  Amit|        HR| 58000|    Delhi|
|     7| Meera|   Finance| 82000|Bangalore|
|     8|  Ravi|        IT| 72000|Hyderabad|
|     9|  Neha|        HR| 61000|  Chennai|
|    10|Vikram|   Finance| 90000|    Delhi|
|    11| Anita|        IT| 65000|Bangalore|
|    12| Manoj|        HR| 62000|   Mumbai|
|    13| Divya|        IT| 77000|Hyderabad|
|    14|Sanjay|   Finance| 85000|  Chennai|
|    15| Pooja|        IT| 69000|Bangalore|
|    16| Kunal|        HR| 61000|    Delhi|
|    17| Sonal|   Finance| 88000|   Mumbai|
|    18|Deepak|        IT| 73000|Hyderabad|
|    19|  Ritu|        HR| 60000|  Chennai|
|    20| Akash|   Finance| 91000

Total employees

In [11]:
emp_df.count()

20

IT employees

In [12]:
emp_df.filter(col("department") == "IT").show()

+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|     1| Rahul|        IT| 70000|Hyderabad|
|     3| Arjun|        IT| 75000|  Chennai|
|     5| Karan|        IT| 50000|   Mumbai|
|     8|  Ravi|        IT| 72000|Hyderabad|
|    11| Anita|        IT| 65000|Bangalore|
|    13| Divya|        IT| 77000|Hyderabad|
|    15| Pooja|        IT| 69000|Bangalore|
|    18|Deepak|        IT| 73000|Hyderabad|
+------+------+----------+------+---------+



Salary > 75k

In [13]:
emp_df.filter(col("salary") > 75000).show()

+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|     4| Priya|   Finance| 80000|Hyderabad|
|     7| Meera|   Finance| 82000|Bangalore|
|    10|Vikram|   Finance| 90000|    Delhi|
|    13| Divya|        IT| 77000|Hyderabad|
|    14|Sanjay|   Finance| 85000|  Chennai|
|    17| Sonal|   Finance| 88000|   Mumbai|
|    20| Akash|   Finance| 91000|    Delhi|
+------+------+----------+------+---------+



Average salary

In [14]:
emp_df.select(avg("salary")).show()

+-----------+
|avg(salary)|
+-----------+
|    71450.0|
+-----------+



Highest paid

In [15]:
emp_df.orderBy(desc("salary")).show(1)

+------+-----+----------+------+-----+
|emp_id| name|department|salary| city|
+------+-----+----------+------+-----+
|    20|Akash|   Finance| 91000|Delhi|
+------+-----+----------+------+-----+
only showing top 1 row


Lowest paid

In [16]:
emp_df.orderBy("salary").show(1)

+------+-----+----------+------+------+
|emp_id| name|department|salary|  city|
+------+-----+----------+------+------+
|     5|Karan|        IT| 50000|Mumbai|
+------+-----+----------+------+------+
only showing top 1 row


Employees per department

In [17]:
emp_df.groupBy("department").count().show()

+----------+-----+
|department|count|
+----------+-----+
|        HR|    6|
|   Finance|    6|
|        IT|    8|
+----------+-----+



Avg salary per department

In [18]:
emp_df.groupBy("department").agg(avg("salary")).show()

+----------+------------------+
|department|       avg(salary)|
+----------+------------------+
|        HR|60333.333333333336|
|   Finance|           86000.0|
|        IT|           68875.0|
+----------+------------------+



Employees per city

In [19]:
emp_df.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    4|
|  Chennai|    4|
|   Mumbai|    3|
|    Delhi|    4|
|Hyderabad|    5|
+---------+-----+



Top 5 salaries

In [20]:
emp_df.orderBy(desc("salary")).show(5)

+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|    20| Akash|   Finance| 91000|    Delhi|
|    10|Vikram|   Finance| 90000|    Delhi|
|    17| Sonal|   Finance| 88000|   Mumbai|
|    14|Sanjay|   Finance| 85000|  Chennai|
|     7| Meera|   Finance| 82000|Bangalore|
+------+------+----------+------+---------+
only showing top 5 rows


Hyderabad employees salary > 70k

In [21]:
emp_df.filter((col("city")=="Hyderabad") & (col("salary")>70000)).show()

+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|     4| Priya|   Finance| 80000|Hyderabad|
|     8|  Ravi|        IT| 72000|Hyderabad|
|    13| Divya|        IT| 77000|Hyderabad|
|    18|Deepak|        IT| 73000|Hyderabad|
+------+------+----------+------+---------+



# DATASET 3 — CSV (sales.csv)

In [25]:
sales_df = spark.read.csv("sales.csv", header=True, inferSchema=True)
sales_df.show()

+-------+------+--------+--------+-----+
|sale_id|emp_id| product|quantity|price|
+-------+------+--------+--------+-----+
|      1|     1|  Laptop|       1|75000|
|      2|     2|   Mouse|       3|  500|
|      3|     3|Keyboard|       2| 1500|
|      4|     1| Monitor|       1|12000|
|      5|     4|  Laptop|       1|75000|
|      6|     3|   Mouse|       2|  500|
|      7|     5|Keyboard|       1| 1500|
|      8|     1|  Laptop|       1|75000|
|      9|     6|   Mouse|       4|  500|
|     10|     7| Monitor|       2|12000|
|     11|     8|Keyboard|       2| 1500|
|     12|     9|   Mouse|       3|  500|
|     13|    10|  Laptop|       1|75000|
|     14|    11| Monitor|       1|12000|
|     15|    12|Keyboard|       2| 1500|
|     16|    13|  Laptop|       1|75000|
|     17|    14|   Mouse|       3|  500|
|     18|    15| Monitor|       1|12000|
|     19|    16|Keyboard|       1| 1500|
|     20|    17|  Laptop|       1|75000|
+-------+------+--------+--------+-----+



Revenue per sales


In [26]:
sales_df = sales_df.withColumn("revenue", col("quantity") * col("price"))
sales_df.show()

+-------+------+--------+--------+-----+-------+
|sale_id|emp_id| product|quantity|price|revenue|
+-------+------+--------+--------+-----+-------+
|      1|     1|  Laptop|       1|75000|  75000|
|      2|     2|   Mouse|       3|  500|   1500|
|      3|     3|Keyboard|       2| 1500|   3000|
|      4|     1| Monitor|       1|12000|  12000|
|      5|     4|  Laptop|       1|75000|  75000|
|      6|     3|   Mouse|       2|  500|   1000|
|      7|     5|Keyboard|       1| 1500|   1500|
|      8|     1|  Laptop|       1|75000|  75000|
|      9|     6|   Mouse|       4|  500|   2000|
|     10|     7| Monitor|       2|12000|  24000|
|     11|     8|Keyboard|       2| 1500|   3000|
|     12|     9|   Mouse|       3|  500|   1500|
|     13|    10|  Laptop|       1|75000|  75000|
|     14|    11| Monitor|       1|12000|  12000|
|     15|    12|Keyboard|       2| 1500|   3000|
|     16|    13|  Laptop|       1|75000|  75000|
|     17|    14|   Mouse|       3|  500|   1500|
|     18|    15| Mon

Total revenue

In [27]:
sales_df.select(sum("revenue")).show()

+------------+
|sum(revenue)|
+------------+
|      529500|
+------------+



Revenue per product

In [28]:
sales_df.groupBy("product").agg(sum("revenue")).show()

+--------+------------+
| product|sum(revenue)|
+--------+------------+
|  Laptop|      450000|
|   Mouse|        7500|
|Keyboard|       12000|
| Monitor|       60000|
+--------+------------+



Quantity per product

In [29]:
sales_df.groupBy("product").agg(sum("quantity")).show()

+--------+-------------+
| product|sum(quantity)|
+--------+-------------+
|  Laptop|            6|
|   Mouse|           15|
|Keyboard|            8|
| Monitor|            5|
+--------+-------------+



Best selling product

In [30]:
sales_df.groupBy("product") \
        .agg(sum("quantity").alias("total_qty")) \
        .orderBy(desc("total_qty")) \
        .show(1)

+-------+---------+
|product|total_qty|
+-------+---------+
|  Mouse|       15|
+-------+---------+
only showing top 1 row


Highest revenue employee

In [31]:
sales_df.groupBy("emp_id") \
        .agg(sum("revenue").alias("total_revenue")) \
        .orderBy(desc("total_revenue")) \
        .show(1)

+------+-------------+
|emp_id|total_revenue|
+------+-------------+
|     1|       162000|
+------+-------------+
only showing top 1 row


Average sale value

In [32]:
sales_df.select(avg("revenue")).show()

+------------+
|avg(revenue)|
+------------+
|     26475.0|
+------------+



Products revenue > 100000

In [33]:
sales_df.groupBy("product") \
        .agg(sum("revenue").alias("total")) \
        .filter(col("total") > 100000) \
        .show()

+-------+------+
|product| total|
+-------+------+
| Laptop|450000|
+-------+------+



# DATASET 4 — JSON (orders.json)

In [34]:
orders_df = spark.read.option("multiline","true").json("orders.json")
orders_df = orders_df.select(explode("orders").alias("data")).select("data.*")
orders_df.show()

+------+---------+--------+--------+--------+
|amount|     city|customer|order_id| product|
+------+---------+--------+--------+--------+
| 75000|Hyderabad|   Rahul|       1|  Laptop|
|  1500|Bangalore|   Sneha|       2|   Mouse|
|  3000|  Chennai|   Arjun|       3|Keyboard|
| 75000|Hyderabad|   Priya|       4|  Laptop|
| 12000|   Mumbai|   Karan|       5| Monitor|
|  1000|Hyderabad|   Rahul|       6|   Mouse|
| 75000|Bangalore|   Sneha|       7|  Laptop|
|  3000|  Chennai|   Arjun|       8|Keyboard|
|  2000|Hyderabad|   Priya|       9|   Mouse|
| 12000|Hyderabad|   Rahul|      10| Monitor|
+------+---------+--------+--------+--------+



Print orders

In [35]:
orders_df.show()

+------+---------+--------+--------+--------+
|amount|     city|customer|order_id| product|
+------+---------+--------+--------+--------+
| 75000|Hyderabad|   Rahul|       1|  Laptop|
|  1500|Bangalore|   Sneha|       2|   Mouse|
|  3000|  Chennai|   Arjun|       3|Keyboard|
| 75000|Hyderabad|   Priya|       4|  Laptop|
| 12000|   Mumbai|   Karan|       5| Monitor|
|  1000|Hyderabad|   Rahul|       6|   Mouse|
| 75000|Bangalore|   Sneha|       7|  Laptop|
|  3000|  Chennai|   Arjun|       8|Keyboard|
|  2000|Hyderabad|   Priya|       9|   Mouse|
| 12000|Hyderabad|   Rahul|      10| Monitor|
+------+---------+--------+--------+--------+



Count orders

In [36]:
orders_df.count()

10

Total sales

In [37]:
orders_df.select(sum("amount")).show()

+-----------+
|sum(amount)|
+-----------+
|     259500|
+-----------+



Spending per customer

In [38]:
orders_df.groupBy("customer").agg(sum("amount")).show()

+--------+-----------+
|customer|sum(amount)|
+--------+-----------+
|   Sneha|      76500|
|   Priya|      77000|
|   Rahul|      88000|
|   Arjun|       6000|
|   Karan|      12000|
+--------+-----------+



Highest spender

In [39]:
orders_df.groupBy("customer") \
         .agg(sum("amount").alias("total")) \
         .orderBy(desc("total")) \
         .show(1)

+--------+-----+
|customer|total|
+--------+-----+
|   Rahul|88000|
+--------+-----+
only showing top 1 row


Sales per product

In [40]:
orders_df.groupBy("product").agg(sum("amount")).show()

+--------+-----------+
| product|sum(amount)|
+--------+-----------+
|  Laptop|     225000|
|   Mouse|       4500|
|Keyboard|       6000|
| Monitor|      24000|
+--------+-----------+



Hyderabad customers

In [41]:
orders_df.filter(col("city")=="Hyderabad") \
         .select("customer") \
         .distinct() \
         .show()

+--------+
|customer|
+--------+
|   Priya|
|   Rahul|
+--------+



Orders > 10k

In [42]:
orders_df.filter(col("amount") > 10000).show()

+------+---------+--------+--------+-------+
|amount|     city|customer|order_id|product|
+------+---------+--------+--------+-------+
| 75000|Hyderabad|   Rahul|       1| Laptop|
| 75000|Hyderabad|   Priya|       4| Laptop|
| 12000|   Mumbai|   Karan|       5|Monitor|
| 75000|Bangalore|   Sneha|       7| Laptop|
| 12000|Hyderabad|   Rahul|      10|Monitor|
+------+---------+--------+--------+-------+



Orders per city

In [43]:
orders_df.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|  Chennai|    2|
|   Mumbai|    1|
|Hyderabad|    5|
+---------+-----+



1. Join employees + sales

In [44]:
joined_df = emp_df.join(sales_df, on="emp_id")
joined_df.show()

+------+------+----------+------+---------+-------+--------+--------+-----+-------+
|emp_id|  name|department|salary|     city|sale_id| product|quantity|price|revenue|
+------+------+----------+------+---------+-------+--------+--------+-----+-------+
|     1| Rahul|        IT| 70000|Hyderabad|      8|  Laptop|       1|75000|  75000|
|     1| Rahul|        IT| 70000|Hyderabad|      4| Monitor|       1|12000|  12000|
|     1| Rahul|        IT| 70000|Hyderabad|      1|  Laptop|       1|75000|  75000|
|     2| Sneha|        HR| 60000|Bangalore|      2|   Mouse|       3|  500|   1500|
|     3| Arjun|        IT| 75000|  Chennai|      6|   Mouse|       2|  500|   1000|
|     3| Arjun|        IT| 75000|  Chennai|      3|Keyboard|       2| 1500|   3000|
|     4| Priya|   Finance| 80000|Hyderabad|      5|  Laptop|       1|75000|  75000|
|     5| Karan|        IT| 50000|   Mumbai|      7|Keyboard|       1| 1500|   1500|
|     6|  Amit|        HR| 58000|    Delhi|      9|   Mouse|       4|  500| 

2.Revenue per employee



In [45]:
emp_revenue = joined_df.groupBy("name") \
                       .agg(sum("revenue").alias("total_revenue"))

emp_revenue.show()

+------+-------------+
|  name|total_revenue|
+------+-------------+
| Kunal|         1500|
| Sonal|        75000|
| Divya|        75000|
|  Ravi|         3000|
|Sanjay|         1500|
| Meera|        24000|
| Sneha|         1500|
| Priya|        75000|
|Vikram|        75000|
| Rahul|       162000|
| Anita|        12000|
| Manoj|         3000|
| Pooja|        12000|
| Arjun|         4000|
|  Amit|         2000|
|  Neha|         1500|
| Karan|         1500|
+------+-------------+



3. Top 5 employees

In [46]:
emp_revenue.orderBy(desc("total_revenue")).show(5)

+------+-------------+
|  name|total_revenue|
+------+-------------+
| Rahul|       162000|
| Priya|        75000|
| Divya|        75000|
| Sonal|        75000|
|Vikram|        75000|
+------+-------------+
only showing top 5 rows


4. Department highest revenue

In [47]:
joined_df.groupBy("department") \
         .agg(sum("revenue").alias("dept_revenue")) \
         .orderBy(desc("dept_revenue")) \
         .show(1)

+----------+------------+
|department|dept_revenue|
+----------+------------+
|        IT|      269500|
+----------+------------+
only showing top 1 row


In [48]:
emp_revenue.write.csv("final_sales_report.csv", header=True)